# Spark-X2.6-4B-En4B-A1B (`v0.3` / `v03_r2_3phase`) — Standalone 72-Layer Inference & Evaluation Suite

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BaiTian6641/Spark-X2.6-Recursion/blob/main/SparkX_Loop_v03_En4B_A1B_Inference_Eval_Colab.ipynb)

This standalone **inference & evaluation notebook** loads the trained **`TassP0lar/Spark-X2.6-4B-En4B-A1B`** (`v03_r2_3phase`, Global Step 208, `ce_loss = 0.4142`) checkpoint directly from Hugging Face Hub and runs:
1. **Cell 1**: Environment & Repository Setup (`Spark-X2.6-Recursion`)
2. **Cell 2**: One-Call 72-Layer Model Load (`8.34B` Total / `5.33B` Active) + Trained `mHC` & `MoE-Engram` Gate Telemetry Audit
3. **Cell 3**: **3-Mode Ablation & Collapse Audit** (`72L Full Loop + Engram` vs `72L Loop Only` vs `36L Base Backbone`) on unseen algorithmic synthesis (`dijkstra_grid`)
4. **Cell 4**: **Out-of-Distribution Coding & STEM Verification Suite** ($O(1)$ TTL-LRU Cache + Exact Sliding-Window Substring)
5. **Cell 5**: **Unclamped 32K Agentic 3D WebGL Synthesis** (Out-of-distribution `Neon Horizon 2099` Hovercar Simulator & `Sakura Voxel Garden` with live `<iframe>` rendering)
6. **Cell 6**: **Interactive 72-Layer Prompt Playground** (`ask_sparkx_72l(...)`)

In [ ]:
# Cell 1: Environment & Repository Setup (Inference-Only)
import os, subprocess, sys
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
REPO_DIR = Path('/content/Spark-X2.6-Recursion')

if Path('/content').exists():
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
    os.chdir(REPO_DIR)
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.48.0', 'accelerate>=1.2.0', 'safetensors>=0.4.5', 'huggingface_hub>=0.27.0'
], check=True)

import torch
print(f'PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name} | Total VRAM: {props.total_memory / (1024**3):.1f} GB')

In [ ]:
# Cell 2: Load Trained Spark-X2.6-4B-En4B-A1B (72L, 32-Expert Top-7) & Inspect Gate Telemetry
import importlib, os, subprocess, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core, load_inference_v03, sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(load_inference_v03)
importlib.reload(sparkx_loop_v03_train_env)

from load_inference_v03 import load_sparkx_loop_v03
from sparkx_loop_v03_core import (
    SparkXLoopV03,
    patch_spark_for_unsloth_triton,
    verify_birkhoff_sinkhorn,
    verify_ngram_causality,
)

HF_REPO_ID = 'TassP0lar/Spark-X2.6-4B-En4B-A1B'
HF_TOKEN = os.environ.get('HF_TOKEN') or None

if 'model' in globals() and model is not None and 'tokenizer' in globals() and tokenizer is not None:
    print('[Hot-Patch] Reusing in-memory 8.23B model and binding latest Transformers 5.x compatibility patches...')
    for _attr_name, _attr_val in SparkXLoopV03.__dict__.items():
        if callable(_attr_val) or isinstance(_attr_val, property):
            setattr(type(model), _attr_name, _attr_val)
    patch_spark_for_unsloth_triton(model.base)
else:
    model, tokenizer = load_sparkx_loop_v03(
        repo_id=HF_REPO_ID,
        device='cuda' if torch.cuda.is_available() else 'cpu',
        dtype=torch.bfloat16,
        offload_tier1b_cpu=False,  # Keep full 8.23B in GPU HBM on A100/H100 (set True on 16-24GB GPUs)
        token=HF_TOKEN,
    )

# Print detailed per-pass mHC and per-anchor MoE-Engram gate telemetry
print('\n' + '=' * 96)
print('LEARNED 72-LAYER mHC RECURRENCE & 8-ANCHOR MoE-ENGRAM TELEMETRY')
print('=' * 96)
p2_attn = [c.attn_extra_gain.detach().float().tanh().abs().item() for c in model.controllers.values()]
p2_ffn  = [c.ffn_extra_gain.detach().float().tanh().abs().item() for c in model.controllers.values()]
p3_attn = [c.p3_attn_extra_gain.detach().float().tanh().abs().item() for idx, c in model.controllers.items() if 12 <= int(idx) < 24]
p3_ffn  = [c.p3_ffn_extra_gain.detach().float().tanh().abs().item() for idx, c in model.controllers.items() if 12 <= int(idx) < 24]
print(f'  Pass-2 [4..28)  Mean |tanh(gain)| : Attn={sum(p2_attn)/len(p2_attn):.4f} | FFN={sum(p2_ffn)/len(p2_ffn):.4f}')
print(f'  Pass-3 [12..24) Mean |tanh(gain)| : Attn={sum(p3_attn)/len(p3_attn):.4f} | FFN={sum(p3_ffn)/len(p3_ffn):.4f}')

for idx, mod in model.iter_engram_anchors():
    g = mod.gate.detach().float().tanh().abs().item()
    p_norm = mod.proj.weight.detach().float().norm().item()
    temp = mod.router_log_temp.detach().float().exp().item()
    scales = torch.softmax(mod.ngram_order_logits.detach().float(), dim=0).tolist()
    role = 'Prelude' if idx == 3 else 'Core/Coda'
    print(
        f'  Anchor Layer {idx:2d} ({role:9s}): gate={g:.4f} | proj_norm={p_norm:5.2f} | '
        f'experts={mod.active_experts}(Top-{mod.active_top_k}) | temp={temp:.3f} | '
        f'ngram_scales=[2g:{scales[0]:.2f}, 3g:{scales[1]:.2f}, skip:{scales[2]:.2f}]'
    )

sinkhorn_chk = verify_birkhoff_sinkhorn()
causal_chk = verify_ngram_causality(model.engram)
print('\n[Mathematical Verification]')
print(f'  Birkhoff-Sinkhorn doubly-stochastic error : row={sinkhorn_chk["max_row_err"]:.2e}, col={sinkhorn_chk["max_col_err"]:.2e}')
print(f'  Causal N-gram future-masking invariance   : prefix_unchanged={causal_chk["causal_prefix_unchanged"]}')

In [ ]:
# Cell 3: Quantitative MoE-Engram Perplexity Audit + 3-Mode Generation Ablation (72L Full vs 72L No-Engram vs 36L Base)
import importlib, math, subprocess, torch
import torch.nn.functional as F
subprocess.run(["git", "pull", "--ff-only"], check=False)
import sparkx_loop_v03_core, sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
from sparkx_loop_v03_train_env import SparkXAgenticWorkspace, _hotbind_v03_runtime_methods

_hotbind_v03_runtime_methods(model)
device = next(model.base.parameters()).device

# Part A: 10-Second Quantitative Perplexity, Next-Token Accuracy & Logit-Delta Audit
eval_texts = [
    "<|im_start|>user\nWrite a Python 3 function using collections.OrderedDict and heapq to merge k sorted streams with TTL expiration.<|im_end|>\n"
    "<|im_start|>assistant\n<think>\n1. Use a min-heap storing (value, timestamp, stream_idx, item_idx).\n2. Filter expired items where now >= timestamp + ttl.\n</think>\n"
    "```python\nimport heapq\nfrom collections import OrderedDict\n\ndef merge_ttl_streams(streams, ttl: float, now: float):\n    heap = []\n    out = []\n    for s_idx, stream in enumerate(streams):\n        if stream:\n            val, ts = stream[0]\n            heapq.heappush(heap, (val, ts, s_idx, 0))\n    while heap:\n        val, ts, s_idx, i_idx = heapq.heappop(heap)\n        if now - ts < ttl:\n            out.append(val)\n        if i_idx + 1 < len(streams[s_idx]):\n            nv, nts = streams[s_idx][i_idx + 1]\n            heapq.heappush(heap, (nv, nts, s_idx, i_idx + 1))\n    return out\n```<|im_end|>",
    "<|im_start|>user\nInitialize a Three.js r128 WebGLRenderer with FogExp2, PerspectiveCamera, and an InstancedMesh of 1000 neon cubes.<|im_end|>\n"
    "<|im_start|>assistant\n<think>\n1. Create Scene, FogExp2, PerspectiveCamera, and WebGLRenderer.\n2. Use THREE.InstancedMesh with Object3D dummy matrix updates.\n</think>\n"
    "```javascript\nconst scene = new THREE.Scene();\nscene.fog = new THREE.FogExp2(0x070312, 0.008);\nconst camera = new THREE.PerspectiveCamera(60, window.innerWidth / window.innerHeight, 0.1, 500);\nconst renderer = new THREE.WebGLRenderer({ antialias: true });\nrenderer.setSize(window.innerWidth, window.innerHeight);\ndocument.body.appendChild(renderer.domElement);\nconst geo = new THREE.BoxGeometry(1, 1, 1);\nconst mat = new THREE.MeshStandardMaterial({ color: 0x00f0ff, emissive: 0xff007f });\nconst inst = new THREE.InstancedMesh(geo, mat, 1000);\nscene.add(inst);\n```<|im_end|>",
    "<|im_start|>user\nProve that a 2x2 doubly-stochastic matrix M = [[p, 1-p], [1-p, p]] has eigenvalues 1 and 2p-1.<|im_end|>\n"
    "<|im_start|>assistant\n<think>\n1. Compute trace(M) = 2p and det(M) = p^2 - (1-p)^2 = 2p - 1.\n2. Solve characteristic polynomial lambda^2 - 2p*lambda + (2p - 1) = 0.\n</think>\n"
    "For $M = \\begin{pmatrix} p & 1-p \\\\ 1-p & p \\end{pmatrix}$, each row sums to $1$, so $\\lambda_1 = 1$. Since $\\text{tr}(M) = \\lambda_1 + \\lambda_2 = 2p$, the second eigenvalue is $\\lambda_2 = 2p - 1 \\in (-1, 1)$ for $p \\in (0, 1)$, bounding $\\|M\\|_2 = 1$.<|im_end|>",
]

configs = [
    ("36L_Base_Only (No Loop, No Engram)", dict(start=4, end=28, passes=1, use_mhc=False, use_engram=False, enable_p3=False), 0.0),
    ("72L_Loop_mHC_Only (Engram OFF)",     dict(start=4, end=28, passes=2, use_mhc=True,  use_engram=False, enable_p3=True, p3_start=12, p3_end=24), 0.0),
    ("72L_Full + MoE-Engram (scale=0.25)", dict(start=4, end=28, passes=2, use_mhc=True,  use_engram=True,  enable_p3=True, p3_start=12, p3_end=24), 0.25),
    ("72L_Full + MoE-Engram (scale=1.00)", dict(start=4, end=28, passes=2, use_mhc=True,  use_engram=True,  enable_p3=True, p3_start=12, p3_end=24), 1.00),
]

print("=" * 104)
print("[QUANTITATIVE MoE-ENGRAM & 72L mHC ABLATION] Cross-Entropy, Perplexity & Next-Token Accuracy")
print("=" * 104)
print(f"{'Configuration':38s} | {'Mean CE Loss':12s} | {'Perplexity':10s} | {'Top-1 Acc':9s} | {'Max Logit Diff vs 72L-NoEng'}")
print("-" * 104)

ref_logits_list = []
with torch.no_grad():
    for name, loop_cfg, eng_scale in configs:
        model.set_loop(**loop_cfg)
        for m in model.all_engram_modules():
            m.decode_engram_scale = eng_scale
        total_nll, total_toks, total_correct, max_diff = 0.0, 0, 0, 0.0
        cur_logits_list = []
        for idx, txt in enumerate(eval_texts):
            ids = tokenizer(txt, return_tensors="pt").input_ids.to(device)
            orig_gates = [m.gate.data.clone() for m in model.all_engram_modules()]
            if eng_scale not in (0.0, 1.0):
                for m in model.all_engram_modules():
                    m.gate.data.mul_(eng_scale)
            logits = model.forward_logits(ids).float()
            for m, g_orig in zip(model.all_engram_modules(), orig_gates):
                m.gate.data.copy_(g_orig)
            cur_logits_list.append(logits)
            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = ids[:, 1:].contiguous()
            nll = F.cross_entropy(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1), reduction="sum")
            preds = shift_logits.argmax(dim=-1)
            total_correct += int((preds == shift_labels).sum().item())
            total_nll += float(nll.item())
            total_toks += int(shift_labels.numel())
            if ref_logits_list:
                max_diff = max(max_diff, float((logits - ref_logits_list[idx]).abs().max().item()))
        if "Engram OFF" in name:
            ref_logits_list = cur_logits_list
        mean_ce = total_nll / max(1, total_toks)
        ppl = math.exp(min(20.0, mean_ce))
        acc = 100.0 * total_correct / max(1, total_toks)
        print(f"{name:38s} | {mean_ce:12.5f} | {ppl:10.4f} | {acc:8.2f}% | {max_diff:8.4f}")
    for m in model.all_engram_modules():
        m.decode_engram_scale = 0.25
print("=" * 104 + "\n")

# Part B: 3-Mode Autoregressive Generation & Collapse Audit (`dijkstra_grid`, capped at 1024 tokens so 36L Base fails fast)
workspace = SparkXAgenticWorkspace("/content/sparkx_v03_eval_workspace", model=model, tokenizer=tokenizer)
ablation_results = workspace.run_collapse_and_ablation_audit(max_new_tokens=1024)


In [ ]:
# Cell 4: Out-of-Distribution Coding & STEM Benchmark Suite (Automated Python Execution + Agentic Self-Repair)
import importlib, re, subprocess, time, torch
subprocess.run(['git', 'pull', '--ff-only'], check=False)
import sparkx_loop_v03_core, sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
from sparkx_loop_v03_train_env import (
    SparkXAgenticWorkspace,
    _hotbind_v03_runtime_methods,
    extract_code_from_response,
)

_hotbind_v03_runtime_methods(model)
if 'workspace' in globals() and workspace is not None:
    for _k, _v in SparkXAgenticWorkspace.__dict__.items():
        if callable(_v):
            setattr(type(workspace), _k, _v)
else:
    workspace = SparkXAgenticWorkspace('/content/sparkx_v03_eval_workspace', model=model, tokenizer=tokenizer)

model.set_loop(start=4, end=28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
device = next(model.base.parameters()).device

SYS_PROMPT = (
    'You are a precise software engineering assistant. '
    'Reason concisely inside <think>...</think>, close </think>, and provide the exact requested Python code.'
)

ood_benchmarks = [
    {
        'name': 'OOD_SWE_TTL_LRU_Cache',
        'prompt': (
            'Write a self-contained Python 3 class `TTLLRUCache` using `collections.OrderedDict` that implements a '
            'Least-Recently-Used cache with per-key expiration timestamps:\n'
            '- `def __init__(self, capacity: int):` initializes `self.capacity = capacity` and an `OrderedDict` storing `(value, expiry)` per key.\n'
            '- `def get(self, key: int, now: float) -> int:` if `key` is not in cache, return `-1`. If `now >= expiry`, delete `key` and return `-1`. Otherwise move `key` to end (most recent) and return `value`.\n'
            '- `def put(self, key: int, value: int, ttl: float, now: float) -> None:` first purge all keys where `now >= expiry`. If `key` is already in cache, remove it. If `len(cache) >= self.capacity`, pop the oldest item (`last=False`). Insert `key` with `(value, now + ttl)`.\n'
            'Output your implementation inside a single ```python ... ``` block.'
        ),
        'test_code': (
            'c = TTLLRUCache(2)\n'
            'c.put(1, 10, ttl=5.0, now=100.0)\n'
            'c.put(2, 20, ttl=10.0, now=101.0)\n'
            'assert c.get(1, now=102.0) == 10\n'
            'c.put(3, 30, ttl=5.0, now=103.0)\n'
            'assert c.get(2, now=103.5) == -1\n'
            'assert c.get(1, now=104.9) == 10\n'
            'assert c.get(1, now=105.1) == -1\n'
            'print("TTL_LRU_OK")\n'
        ),
        'expected_stdout': 'TTL_LRU_OK',
    },
    {
        'name': 'OOD_Algorithm_MinWindowSubstring',
        'prompt': (
            'Write a self-contained Python 3 function `def min_window_substring(s: str, t: str) -> str:` that finds the minimum '
            'contiguous substring of `s` containing all characters of `t` (including multiplicity) using `collections.Counter`:\n'
            '- If `not s or not t`, return `""`.\n'
            '- Let `need = Counter(t)`, `window = Counter()`, `required = len(need)`, `formed = 0`, `left = 0`, and `ans = (float("inf"), 0, 0)`.\n'
            '- Loop `for right, ch in enumerate(s):` increment `window[ch] += 1`; if `ch in need and window[ch] == need[ch]: formed += 1`.\n'
            '- Inside that loop, `while left <= right and formed == required:` if `right - left + 1 < ans[0]: ans = (right - left + 1, left, right + 1)`; then decrement `window[s[left]] -= 1`, if `s[left] in need and window[s[left]] < need[s[left]]: formed -= 1`, and increment `left += 1`.\n'
            '- Return `"" if ans[0] == float("inf") else s[ans[1]:ans[2]]`.\n'
            'Output your implementation inside a single ```python ... ``` block.'
        ),
        'test_code': (
            'g1 = min_window_substring("ADOBECODEBANC", "ABC")\n'
            'assert g1 == "BANC", f"Expected \'BANC\', got {g1!r}"\n'
            'g2 = min_window_substring("a", "aa")\n'
            'assert g2 == "", f"Expected \'\', got {g2!r}"\n'
            'g3 = min_window_substring("aaflslflsldkalskaaa", "aaa")\n'
            'assert g3 == "aaa", f"Expected \'aaa\', got {g3!r}"\n'
            'print("MIN_WINDOW_OK")\n'
        ),
        'expected_stdout': 'MIN_WINDOW_OK',
    },
]

for bench in ood_benchmarks:
    print('\n' + '-' * 96)
    print(f"Running {bench['name']} (72L Full Loop + 32-Expert Top-7 Engram)...")
    messages = [
        {'role': 'system', 'content': SYS_PROMPT},
        {'role': 'user', 'content': bench['prompt']},
    ]
    passed = False
    for turn in range(1, 3):
        _enc = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors='pt')
        p_ids = (_enc.input_ids if hasattr(_enc, 'input_ids') else _enc).to(device)
        t0 = time.time()
        out_ids = model.generate_cached(
            p_ids,
            max_new_tokens=2048,
            temperature=0.2,
            top_p=0.90,
            repetition_penalty=1.03,
            eos_token_id=tokenizer.eos_token_id,
            tokenizer=tokenizer,
            stream_print=True,
        )
        dt = max(1e-3, time.time() - t0)
        gen_ids = out_ids[0, p_ids.shape[1]:]
        text = tokenizer.decode(gen_ids, skip_special_tokens=True)

        body = text.split('</think>', 1)[-1].strip() if '</think>' in text else text.strip()
        cand_blocks = re.findall(r'```(?:python|py)?\s*\n(.*?)```', body, flags=re.DOTALL | re.IGNORECASE)
        if not cand_blocks:
            cand_blocks = [extract_code_from_response(text, target_ext='.py')]

        res = {'ok': False, 'stdout': '', 'stderr': 'No code block found'}
        for raw_blk in cand_blocks:
            fixed_blk = re.sub(r'class\s+(\w+)\s*\([^)]*:[^)]*\)\s*:', r'class \1:', raw_blk)
            res = workspace.execute_python(f'{bench["name"]}.py', fixed_blk + '\n\n' + bench['test_code'])
            if res['ok'] and (bench['expected_stdout'] in res['stdout']):
                passed = True
                break

        print(
            f"[{bench['name']} Turn {turn} Result] Passed={passed} | Tokens={gen_ids.numel()} ({gen_ids.numel()/dt:.1f} tok/s) | "
            f"Stdout={res['stdout'].strip()!r}"
        )
        if passed:
            break
        print(f"  [Turn {turn} Stderr]: {res['stderr'].strip()[:300]}")
        messages.append({'role': 'assistant', 'content': text})
        messages.append({
            'role': 'user',
            'content': f"Your code failed verification with stderr:\n{res['stderr'][:600]}\nPlease fix the bug and output the corrected Python code inside ```python ... ```.",
        })

In [ ]:
# Cell 5: Unclamped 32K Agentic 3D WebGL Synthesis & Live Preview
# Set FORCE_REGENERATE = False to instantly hot-repair & render your existing /content/sparkx_v03_eval_workspace/index.html in <0.5s,
# or set FORCE_REGENERATE = True to run a fresh 72-layer generation pass.
import html, importlib, subprocess
from pathlib import Path
subprocess.run(["git", "pull", "--ff-only"], check=False)
import sparkx_loop_v03_core, sparkx_loop_v03_train_env
importlib.reload(sparkx_loop_v03_core)
importlib.reload(sparkx_loop_v03_train_env)
from IPython.display import HTML, display
from sparkx_loop_v03_train_env import (
    DEFAULT_MAX_NEW_TOKENS,
    OOD_NEON_HORIZON_MISSION,
    SparkXAgenticWorkspace,
    _hotbind_v03_runtime_methods,
)

_hotbind_v03_runtime_methods(model)
if "workspace" in globals() and workspace is not None:
    for _k, _v in SparkXAgenticWorkspace.__dict__.items():
        if callable(_v):
            setattr(type(workspace), _k, _v)
else:
    workspace = SparkXAgenticWorkspace("/content/sparkx_v03_eval_workspace", model=model, tokenizer=tokenizer)

MISSION_CHOICE = "neon_horizon_ood"  # Options: "neon_horizon_ood" or "sakura_garden"
FORCE_REGENERATE = False  # Set True to generate from scratch; False hot-repairs existing index.html in <0.5s

existing_html_path = Path("/content/sparkx_v03_eval_workspace/index.html")
if existing_html_path.exists() and not FORCE_REGENERATE:
    print(f"[Fast Hot-Repair] Repairing & upgrading existing {existing_html_path} ({existing_html_path.stat().st_size} bytes)...")
    raw_existing = existing_html_path.read_text(encoding="utf-8")
    res = workspace.write_file("index.html", raw_existing)
    print(f"[Hot-Repair Complete] Saved {res['saved_path']} ({res['bytes']} bytes) | Verification: {res['verification']}")
    workspace.render_inline_colab(res["content"])
else:
    sakura_mission = (
        "Build a complete, self-contained, single-file HTML5 + Three.js r128 3D Interactive Voxel Cherry Blossom "
        "(Sakura) Garden webpage (`<!DOCTYPE html>` ... `</html>`) and save it as `index.html`.\n"
        "Strict Technical & Visual Requirements:\n"
        "1. Load Three.js (`https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js`) and `OrbitControls.js` (`https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js`).\n"
        "2. Camera & Lighting: Set `camera = new THREE.PerspectiveCamera(55, window.innerWidth / window.innerHeight, 0.1, 400)` at `(0, 18, 32)` with `OrbitControls` (`enableDamping = true`, `autoRotate = true`). Add `AmbientLight(0xffd6e8, 0.7)`, `DirectionalLight(0xfff0dd, 1.2)` at `(25, 40, 20)`, and `FogExp2(0x1a1025, 0.012)`.\n"
        "3. Voxel Island, Pond & Bridge: Render a stepped voxel island (`x, z` in `[-14..14]`) with grass/stone voxels (`BoxGeometry(1, 1, 1)`), a central reflective turquoise water pond (`CircleGeometry(7, 32)` rotated `-Math.PI/2` at `y = 0.2`), a red wooden arched bridge across the pond, and a traditional red Torii gate (`CylinderGeometry` pillars + crossbeams).\n"
        "4. Sakura Trees, Lanterns & 500 Drifting Petals: Build 8 gnarled dark-wood cherry blossom trees with layered pink/blush/white voxel canopies (`0xffb7c5`, `0xff8da1`, `0xffe4e9`), 6 glowing stone lanterns (`PointLight(0xffaa44, 1.8, 12)`), and animate 500 falling sakura petals (`THREE.Points` or mesh array) drifting in a gentle sine-wave wind and resetting to `y = 22` when `y < 0`.\n"
        "5. Interactive HUD Buttons: Include visible top-left buttons (`id=\"day\"`, `id=\"sunset\"`, `id=\"night\"`) that smoothly switch sky/fog/light colors between Spring Daylight (`0xdbeeff`), Golden Sunset (`0x3b1d38`), and Lantern Night (`0x0b0716`)."
    )
    chosen_prompt = OOD_NEON_HORIZON_MISSION if MISSION_CHOICE == "neon_horizon_ood" else sakura_mission
    mission_result = workspace.run_agentic_mission(
        chosen_prompt,
        target_filename="index.html",
        max_turns=2,
        max_new_tokens_per_turn=DEFAULT_MAX_NEW_TOKENS,
        run_ablation_first=False,
        use_ood_mission=(MISSION_CHOICE == "neon_horizon_ood"),
    )
    if mission_result.get("html"):
        escaped = html.escape(mission_result["html"], quote=True)
        display(HTML(f"<iframe srcdoc=\"{escaped}\" width=\"100%\" height=\"600\" style=\"border:1px solid #00e5ff; border-radius:10px;\"></iframe>"))


In [ ]:
# Cell 6: Interactive 72-Layer Prompt Playground (with Automated Python Execution Verification)
import re
from sparkx_loop_v03_train_env import SparkXAgenticWorkspace, extract_code_from_response

DEFAULT_SPARKX_72L_SYSTEM_PROMPT = (
    "You are Spark-X2.6-4B-En4B-A1B (v0.3), a 72-layer Looped Transformer (8.23B Total / 5.23B Active parameters) "
    "built on a 36-layer backbone with telescoping recurrence (Prelude[0..4) -> Pass1[4..28) -> Pass2[4..28) -> Pass3[12..24) -> Coda[28..36)).\n"
    "Your two core architectural mechanisms are:\n"
    "1. Two-Stream Birkhoff-Sinkhorn Hyper-Connections (mHC): mixes primary and highway residual streams `x` of shape `[B, S, 2, D]` "
    "using learnable 2x2 logits `self.mix_logits = nn.Parameter(torch.tensor([[2.0, -2.0], [-2.0, 2.0]]))`: "
    "`p = torch.sigmoid(L[0, 0] + L[1, 1] - L[0, 1] - L[1, 0])`, `s0, s1 = x[:, :, 0, :], x[:, :, 1, :]`, "
    "`out = torch.stack([p * s0 + (1.0 - p) * s1, (1.0 - p) * s0 + p * s1], dim=2)`, bounding spectral norm <= 1 across all 72 virtual layers.\n"
    "2. 8-Anchor 3-Tier Zipfian MoE-Engram: causal O(1) multi-head XOR-shift N-gram hashing (2-gram, 3-gram, skip-gram) + 1 shared expert + 32 routed SwiGLU experts (Top-7) injected at every 4th layer.\n"
    "Reason concisely inside <think>...</think>, close </think>, and provide complete, self-contained, runnable Python code."
)

def ask_sparkx_72l(
    prompt: str,
    system_prompt: str = DEFAULT_SPARKX_72L_SYSTEM_PROMPT,
    max_new_tokens: int = 4096,
    temperature: float = 0.25,
    top_p: float = 0.90,
    verify_python: bool = True,
    max_turns: int = 2,
) -> str:
    model.set_loop(start=4, end=28, passes=2, use_mhc=True, use_engram=True, enable_p3=True, p3_start=12, p3_end=24)
    device = next(model.base.parameters()).device
    ws = globals().get("workspace") or SparkXAgenticWorkspace("/content/sparkx_v03_eval_workspace", model=model, tokenizer=tokenizer)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]
    reply = ""
    for turn in range(1, max_turns + 1):
        _enc = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
        input_ids = (_enc.input_ids if hasattr(_enc, "input_ids") else _enc).to(device)
        out_ids = model.generate_cached(
            input_ids,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            repetition_penalty=1.03,
            eos_token_id=tokenizer.eos_token_id,
            tokenizer=tokenizer,
            stream_print=True,
        )
        reply = tokenizer.decode(out_ids[0, input_ids.shape[1]:], skip_special_tokens=True)
        print(f"\n" + "=" * 96 + f"\nMODEL RESPONSE (Turn {turn}):\n" + "=" * 96)
        print(reply)
        if not verify_python:
            break
        body = reply.split("</think>", 1)[-1].strip() if "</think>" in reply else reply.strip()
        py_blocks = re.findall(r"```(?:python|py)\s*\n(.*?)```", body, flags=re.DOTALL | re.IGNORECASE)
        if not py_blocks:
            break
        code_to_run = max(py_blocks, key=len).strip()
        exec_res = ws.execute_python("playground_check.py", code_to_run)
        print("-" * 96)
        print(f"[Python Execution Audit | Turn {turn}] Passed={exec_res['ok']} | Stdout={exec_res['stdout'].strip()!r}")
        if exec_res["ok"]:
            break
        print(f"[Stderr]: {exec_res['stderr'].strip()[:400]}")
        if turn < max_turns:
            messages.append({"role": "assistant", "content": reply})
            messages.append({
                "role": "user",
                "content": (
                    f"Your Python code raised a runtime error when executed:\n{exec_res['stderr'][:600]}\n"
                    "Fix the bug (use `torch.stack([p * s0 + (1 - p) * s1, (1 - p) * s0 + p * s1], dim=2)` instead of `torch.tensor([[p, 1-p], ...])`, "
                    "and verify `torch.norm(y, dim=(-2, -1)) <= torch.norm(x, dim=(-2, -1)) + 1e-5` since doubly-stochastic mixing is a contraction) "
                    "and output the complete corrected explanation and runnable ```python ... ``` script."
                ),
            })
    return reply

# Example interactive test:
_ = ask_sparkx_72l(
    "Explain why two-stream doubly-stochastic Sinkhorn Hyper-Connections (mHC) prevent signal explosion "
    "across 72 recurrent transformer visits, and provide a concise, runnable PyTorch `nn.Module` implementation "
    "of the exact 2x2 closed-form Sinkhorn mixer on tensor shape `[B, S, 2, D]` (using `s0, s1 = x[:, :, 0, :], x[:, :, 1, :]` "
    "and `torch.stack([p * s0 + (1 - p) * s1, (1 - p) * s0 + p * s1], dim=2)`) with a self-test verifying doubly-stochastic row/col sums == 1 and contraction `norm(y) <= norm(x)`."
)
